# Prática - Mapeamento de Texturas + MVP

### Primeiro, vamos importar as bibliotecas necessárias.

In [169]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

In [170]:
from PIL import Image
import os

input_dir = "skybox/skycube2_bmp"
output_dir = "skybox/skycube2_png"
os.makedirs(output_dir, exist_ok=True)

for i in range(1, 7):
    filename = f"skyrender{i:04d}.bmp"
    img = Image.open(os.path.join(input_dir, filename)).convert("RGB")
    img = img.rotate(180)  # rotaciona 180 graus
    img.save(os.path.join(output_dir, f"skyrender{i:04d}.png"))

### Inicializando janela

In [171]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 1280

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [172]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


### Carregando Modelos (vértices e texturas) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [173]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []

def load_obj(objFile):
    modelo = load_model_from_file(objFile)
    
    verticeInicial = len(vertices_list)

    for face in modelo['faces']:
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])

    verticeFinal = len(vertices_list)

    return verticeInicial, verticeFinal - verticeInicial


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r", encoding="latin-1"):
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            if texture_id > 0 and len(modelo['texture']) > 0:
                textures_coord_list.append(modelo['texture'][texture_id - 1])
            else:
                textures_coord_list.append([0.0, 0.0])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    texture_ids = glGenTextures(len(texturesList))

    try:
        texture_ids = list(texture_ids)
    except TypeError:
        texture_ids = [texture_ids]

    for i in range(len(texturesList)):
        load_texture_from_file(texture_ids[i], texturesList[i])

    return verticeInicial, verticeFinal - verticeInicial, texture_ids

### Vamos carregar cada modelo e definir funções para desenhá-los

In [174]:

# --- Piso Interno ---
# vértices do plano (posição XYZ + coordenada de textura UV)
piso_interno_vertices_raw = np.array([
    -1.0,  0.0, -1.0,   0.0,  4.0,
     1.0,  0.0, -1.0,   4.0,  4.0,
     1.0,  0.0,  1.0,   4.0,  0.0,
     1.0,  0.0,  1.0,   4.0,  0.0,
    -1.0,  0.0,  1.0,   0.0,  0.0,
    -1.0,  0.0, -1.0,   0.0,  4.0,
], dtype=np.float32)

# insere os vértices e texturas nas listas globais
verticeInicial_piso_interno = len(vertices_list)

for i in range(0, len(piso_interno_vertices_raw), 5):
    vertices_list.append(piso_interno_vertices_raw[i:i+3].tolist())
    textures_coord_list.append(piso_interno_vertices_raw[i+3:i+5].tolist())
quantosVertices_piso_interno = 6

# carrega a textura
texturas_piso_interno = glGenTextures(1)
load_texture_from_file(texturas_piso_interno, 'objetos/planks/Planks036B_1K-PNG_Color.png')

def desenha_piso_interno(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_piso_interno, quantosVertices_piso_interno)

    
verticeInicial_ground, quantosVertices_ground, texturas_ground = load_obj_and_texture('objetos/ground/Rocks_and_melting_snow.obj', ['objetos/ground/Rocks_and_melting_snow.jpg'])
def desenha_ground(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_ground, quantosVertices_ground) ## renderizando


verticeInicial_house, quantosVertices_house, texturas_house = load_obj_and_texture('objetos/house/Snow covered CottageOBJ.obj', ['objetos/house/Cottage Texture.jpg'])

def desenha_house(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_house, quantosVertices_house) ## renderizando

verticeInicial_christmas_tree, quantosVertices_christmas_tree, texturas_christmas_tree = load_obj_and_texture('objetos/christmas_tree/model.obj', ['objetos/christmas_tree/tex_u1_v1.jpg'])

def desenha_christmas_tree(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_christmas_tree, quantosVertices_christmas_tree) ## renderizando

verticeInicial_snowman, quantosVertices_snowman, texturas_snowman = load_obj_and_texture('objetos/snowman/model.obj', ['objetos/snowman/tex_u1_v1.jpg'])

def desenha_snowman(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_snowman, quantosVertices_snowman) ## renderizando

verticeInicial_sofa, quantosVertices_sofa, texturas_sofa = load_obj_and_texture('objetos/sofa/sofa.obj', ['objetos/sofa/texture.jpg'])

def desenha_sofa(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_sofa, quantosVertices_sofa) ## renderizando

verticeInicial_winter_tree, quantosVertices_winter_tree, texturas_winter_tree = load_obj_and_texture('objetos/winter_tree/winter_tree.obj', ['objetos/winter_tree/textures/winter-tree8.png'])

def desenha_winter_tree(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_winter_tree, quantosVertices_winter_tree) ## renderizando

verticeInicial_zombie, quantosVertices_zombie, texturas_zombie = load_obj_and_texture('objetos/zombie/source/zombie.obj', ['objetos/zombie/textures/zombie_classic_sheet.png'])

def desenha_zombie(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_zombie, quantosVertices_zombie) ## renderizando

verticeInicial_jon_snow, quantosVertices_jon_snow, texturas_jon_snow = load_obj_and_texture('objetos/jon-snow/source/jon-snow.obj', ['objetos/jon-snow/textures/jonsnowDiffuseMap.jpg'])

def desenha_jon_snow(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_jon_snow, quantosVertices_jon_snow) ## renderizando

verticeInicial_fireplace, quantosVertices_fireplace, texturas_fireplace = load_obj_and_texture('objetos/fireplace/fireplace.obj', ['objetos/fireplace/texture.jpg'])

def desenha_fireplace(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_fireplace, quantosVertices_fireplace) ## renderizando


verticeInicial_boneca, quantosVertices_boneca, texturas_boneca = load_obj_and_texture(
    'objetos/boneca/boneca.obj',
    ['objetos/boneca/tex/boneca.jpg']
)

def desenha_boneca(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_boneca, quantosVertices_boneca)

verticeInicial_mesa, quantosVertices_mesa, texturas_mesa = load_obj_and_texture(
    'objetos/Tabel/table.obj',
    ['objetos/Tabel/tex/WoodSeemles.jpg']
)

def desenha_mesa(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_mesa, quantosVertices_mesa)

verticeInicial_lata, quantosVertices_lata, texturas_lata = load_obj_and_texture(
    'objetos/lata/Lata.obj',
    ['objetos/lata/Coca-Cola 01.jpg']
)

def desenha_lata(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_lata, quantosVertices_lata)

verticeInicial_gun, quantosVertices_gun, texturas_gun = load_obj_and_texture(
    'objetos/Gun/Gun.obj',
    ['objetos/Gun/Gun.png']
)

def desenha_gun(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_gun, quantosVertices_gun)


1
Processando modelo objetos/ground/Rocks_and_melting_snow.obj. Vertice inicial: 6
Processando modelo objetos/ground/Rocks_and_melting_snow.obj. Vertice final: 2455920
2
Processando modelo objetos/house/Snow covered CottageOBJ.obj. Vertice inicial: 2455920
Processando modelo objetos/house/Snow covered CottageOBJ.obj. Vertice final: 2458488
3
Processando modelo objetos/christmas_tree/model.obj. Vertice inicial: 2458488
Processando modelo objetos/christmas_tree/model.obj. Vertice final: 3358011
4
Processando modelo objetos/snowman/model.obj. Vertice inicial: 3358011
Processando modelo objetos/snowman/model.obj. Vertice final: 6552009
5
Processando modelo objetos/sofa/sofa.obj. Vertice inicial: 6552009
Processando modelo objetos/sofa/sofa.obj. Vertice final: 6635139
6
Processando modelo objetos/winter_tree/winter_tree.obj. Vertice inicial: 6635139
Processando modelo objetos/winter_tree/winter_tree.obj. Vertice final: 6750375
7
Processando modelo objetos/zombie/source/zombie.obj. Vertice i

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar dois slots (buffers): um para os vértices e outro para as texturas.

In [175]:
buffer_VBO = glGenBuffers(2)

### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [176]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [177]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


In [178]:
# ============================================================
# SKYBOX — setup completo
# ============================================================

# --- 1. Compila os shaders do skybox manualmente ---
def compila_shader(source, shader_type):
    shader = glCreateShader(shader_type)
    glShaderSource(shader, source)
    glCompileShader(shader)
    if not glGetShaderiv(shader, GL_COMPILE_STATUS):
        raise RuntimeError(glGetShaderInfoLog(shader).decode())
    return shader

with open("skybox.vs") as f:
    skybox_vs_src = f.read()
with open("skybox.fs") as f:
    skybox_fs_src = f.read()

skybox_program = glCreateProgram()
glAttachShader(skybox_program, compila_shader(skybox_vs_src, GL_VERTEX_SHADER))
glAttachShader(skybox_program, compila_shader(skybox_fs_src, GL_FRAGMENT_SHADER))
glLinkProgram(skybox_program)

# --- 2. Vértices do cubo ---
skybox_vertices = np.array([
    -1,  1, -1,   -1, -1, -1,    1, -1, -1,
     1, -1, -1,    1,  1, -1,   -1,  1, -1,

    -1, -1,  1,   -1, -1, -1,   -1,  1, -1,
    -1,  1, -1,   -1,  1,  1,   -1, -1,  1,

     1, -1, -1,    1, -1,  1,    1,  1,  1,
     1,  1,  1,    1,  1, -1,    1, -1, -1,

    -1, -1,  1,   -1,  1,  1,    1,  1,  1,
     1,  1,  1,    1, -1,  1,   -1, -1,  1,

    -1,  1, -1,    1,  1, -1,    1,  1,  1,
     1,  1,  1,   -1,  1,  1,   -1,  1, -1,

    -1, -1, -1,   -1, -1,  1,    1, -1, -1,
     1, -1, -1,   -1, -1,  1,    1, -1,  1,
], dtype=np.float32)

# --- 3. VAO e VBO do skybox ---
skybox_vao = glGenVertexArrays(1)
skybox_vbo = glGenBuffers(1)

glBindVertexArray(skybox_vao)
glBindBuffer(GL_ARRAY_BUFFER, skybox_vbo)
glBufferData(GL_ARRAY_BUFFER, skybox_vertices.nbytes, skybox_vertices, GL_STATIC_DRAW)
glEnableVertexAttribArray(0)
glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, 12, ctypes.c_void_p(0))
glBindVertexArray(0)

# --- 4. Carrega as 6 texturas do cubemap ---
skybox_faces = [
    "skybox/skycube2_png/skyrender0004.png",
    "skybox/skycube2_png/skyrender0002.png",
    "skybox/skycube2_png/skyrender0005.png",
    "skybox/skycube2_png/skyrender0006.png",
    "skybox/skycube2_png/skyrender0001.png",
    "skybox/skycube2_png/skyrender0003.png",
]

skybox_texture = glGenTextures(1)
glBindTexture(GL_TEXTURE_CUBE_MAP, skybox_texture)

targets = [
    GL_TEXTURE_CUBE_MAP_POSITIVE_X,
    GL_TEXTURE_CUBE_MAP_NEGATIVE_X,
    GL_TEXTURE_CUBE_MAP_POSITIVE_Y,
    GL_TEXTURE_CUBE_MAP_NEGATIVE_Y,
    GL_TEXTURE_CUBE_MAP_POSITIVE_Z,
    GL_TEXTURE_CUBE_MAP_NEGATIVE_Z,
]


for i, face_path in enumerate(skybox_faces):
    img = Image.open(face_path).convert("RGB")
    if i == 2:  # TOP
        img = img.rotate(270)
    img_data = img.tobytes("raw", "RGB", 0, -1)
    glTexImage2D(targets[i], 0, GL_RGB, img.width, img.height,
                 0, GL_RGB, GL_UNSIGNED_BYTE, img_data)

glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)

# volta para o shader principal
ourShader.use()

def desenha_skybox(mat_view, mat_projection):
    glDepthFunc(GL_LEQUAL)
    glUseProgram(skybox_program)

    # remove translação da view (só rotação)
    view_rot = glm.mat4(glm.mat3(glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp)))
    view_rot = np.array(view_rot)

    glUniformMatrix4fv(glGetUniformLocation(skybox_program, "view"),
                       1, GL_TRUE, view_rot)
    glUniformMatrix4fv(glGetUniformLocation(skybox_program, "projection"),
                       1, GL_TRUE, mat_projection)
    glUniform1i(glGetUniformLocation(skybox_program, "skybox"), 0)

    glBindVertexArray(skybox_vao)
    glActiveTexture(GL_TEXTURE0)
    glBindTexture(GL_TEXTURE_CUBE_MAP, skybox_texture)
    glDrawArrays(GL_TRIANGLES, 0, 36)
    glBindVertexArray(0)

    glDepthFunc(GL_LESS)
    ourShader.use()

### Eventos para modificar a posição da câmera.

* Usei as teclas A, S, D e W para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [179]:
#cameraPos   = glm.vec3(0.0,  0.0,  1.0);
#cameraFront = glm.vec3(0.0,  0.0, -1.0);
#cameraUp    = glm.vec3(0.0,  1.0,  0.0);

# camera
cameraPos   = glm.vec3(0.0, 2.0, -20)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   = -90.0	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2

# transformações interativas
snowman_scale = 0.1      # escala inicial do boneco de neve
jon_snow_angle = -90.0   # ângulo inicial do jon snow
zombie_x = -10.0         # posição x inicial do zombie
zombie_z = -13.0         # posição z inicial do zombie


def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode
    global snowman_scale, jon_snow_angle, zombie_x, zombie_z

    # limites do mundo
    LIMITE_X = 48.0
    LIMITE_Z = 48.0
    ALTURA_MIN = 3   # não afunda no chão
    ALTURA_MAX = 120.0   # não sobe além do céu

    cameraSpeed = 250 * deltaTime

    cameraPos.x = max(-LIMITE_X, min(LIMITE_X, cameraPos.x))
    cameraPos.z = max(-LIMITE_Z, min(LIMITE_Z, cameraPos.z))
    cameraPos.y = max(ALTURA_MIN, min(ALTURA_MAX, cameraPos.y))
    
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    
    cameraSpeed = 250 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    # tecla = (mesmo que + sem shift)
    if key == glfw.KEY_EQUAL and (action == glfw.PRESS or action == glfw.REPEAT):
        snowman_scale += 0.01
    
    # tecla - 
    if key == glfw.KEY_MINUS and (action == glfw.PRESS or action == glfw.REPEAT):
        snowman_scale = max(0.01, snowman_scale - 0.01)
    
    # teclado numérico + 
    if key == glfw.KEY_KP_ADD and (action == glfw.PRESS or action == glfw.REPEAT):
        snowman_scale += 0.01
    
    # teclado numérico -
    if key == glfw.KEY_KP_SUBTRACT and (action == glfw.PRESS or action == glfw.REPEAT):
        snowman_scale = max(0.01, snowman_scale - 0.01)

    # Jon Snow — rotação
    if key == glfw.KEY_R and (action == glfw.PRESS or action == glfw.REPEAT):
        jon_snow_angle += 5.0

    # Zombie — translação
    if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
        zombie_z += 0.5
        
    if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
        zombie_z -= 0.5
        
    if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
        zombie_x -= 0.5
        
    if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
        zombie_x += 0.5
        

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [180]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 1000.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### Nesse momento, nós exibimos a janela!


In [181]:
glfw.show_window(window)

### Loop principal da janela.

In [182]:
glEnable(GL_DEPTH_TEST) ### importante para 3D
polygonal_mode = False 

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)
    
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)

    mat_view = view()
    mat_projection = projection()

    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    desenha_ground(
        0.0,
        0, 0, 1,
        0, -5, 0,     
        50, 1, 50,     
        texturas_ground[0]
    )

    desenha_house(
        0.0,
        0, 0, 1,
        0, 1, 0,       # posição base
        0.15, 0.15, 0.15,
        texturas_house[0]
    )
    
    desenha_snowman(
        0.0,
        0, 1, 0,
        20, 1, 0,
        snowman_scale, snowman_scale, snowman_scale,  # escala dinâmica
        texturas_snowman[0]
    )
    
    desenha_sofa(
        0.0,
        0, 0, 1,
        2, 0.85, 4,       # posição base
        1, 1, 1,
        texturas_sofa[0]
    )
    
    desenha_winter_tree(
        0.0,
        0, 0, 1,
        -10, 1, 0,
        3.5, 3.5, 3.5,  
        texturas_winter_tree[0]
    )
    
    desenha_christmas_tree(
        0.0,
        0, 0, 1,
        -2.25, 1, 5,       # posição base
        0.1, 0.1, 0.1,
        texturas_christmas_tree[0]
    )

    desenha_zombie(
        90.0,
        0, 1, 0,
        zombie_x, 1, zombie_z,  # posição dinâmica
        0.06, 0.055, 0.1,
        texturas_zombie[0]
    )

    desenha_jon_snow(
        jon_snow_angle,  # ângulo dinâmico
        0, 1, 0,
        10.0, 1, -13,
        0.1, 0.08, 0.1,
        texturas_jon_snow[0]
    )

    desenha_fireplace(
        0.0,
        0, 0, 1,
        2.25, 1, 7.0,       # posição base
        1, 1, 1,
        texturas_fireplace[0]
    )
    
    desenha_boneca(
        0.0,
        0, 1, 0,
        2, 1.2, 4,        # posição — ajusta depois
        0.001, 0.001, 0.001,  # escala — ajusta depois
        texturas_boneca[0]
    )

    desenha_mesa(
        90.0,
        0, 1, 0,
        2.1, 1, 5.5,        # entre o sofá e a lareira
        0.008, 0.008, 0.008,
        texturas_mesa[0]
    )

    desenha_lata(
        0.0,
        0, 1, 0,
        1.7, 1.5, 5.3,    # um pouco acima da mesa, lado direito
        0.001, 0.001, 0.001,
        texturas_lata[0]
    )

    desenha_gun(
        90.0,
        1, 0, 0,
        2.2, 1.5, 5.5,   # um pouco acima da mesa, lado esquerdo
        0.45, 0.45, 0.45,
        texturas_gun[0]
    )
        
    desenha_piso_interno(
        0.0,
        0, 0, 1,
        0, 1.15, 0,   
        4, 1, 6.8,      # escala para cobrir o interior
        texturas_piso_interno
    )

    desenha_skybox(mat_view, mat_projection)  
    
    glfw.swap_buffers(window)

glfw.terminate()